# Week 4: Cryptography, Hands-On
**Goal:** break an old cipher by hand, then use real encryption correctly to protect a message.

**Time:** about 2 hours. **You need:** Weeks 1 to 3 (functions, loops, dictionaries, strings).

**Story.** Your team intercepted a scrambled message (`intercepted.txt`) that a club used to "hide" its plans.
First you break it with no key, using only the structure of English. Then you learn why that trick fails against
modern encryption, and you protect a message the right way.

> This is all fictional lab data. Use encryption to protect your own data, never to break into anyone else's.

## Part 0: Get the data (run this cell)

In [ ]:
import os, urllib.request

DATA_URL = "https://raw.githubusercontent.com/Pantea1358/Cyber-labs-/main/weeks/week04-cryptography/data/"
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/intercepted.txt"):
    urllib.request.urlretrieve(DATA_URL + "intercepted.txt", "data/intercepted.txt")

message = open("data/intercepted.txt").read().strip()
print(message)

## Part 1: The Caesar cipher (15 min)
A Caesar cipher shifts every letter by a fixed amount: with shift 3, `a` becomes `d`. Write `caesar(text, shift)`
that shifts letters and leaves spaces and punctuation alone. Encrypting with `shift` and decrypting with `-shift`
should return the original.

*Hint:* for a lowercase letter, `chr((ord(ch) - 97 + shift) % 26 + 97)`.

In [ ]:
def caesar(text, shift):
    out = []
    for ch in text:
        # TODO: shift uppercase letters, lowercase letters; leave everything else unchanged
        out.append(ch)
    return "".join(out)

secret = caesar("attack at dawn", 3)
print("Encrypted:", secret)
print("Decrypted:", caesar(secret, -3))

In [ ]:
assert caesar("abc", 1) == "bcd"
assert caesar("Hello, World!", 5) == "Mjqqt, Btwqi!"
assert caesar(caesar("test message", 9), -9) == "test message"
print("caesar() works!")

## Part 2: Break it by brute force (15 min)
A Caesar cipher has only 25 possible shifts, so you can just try them all. Print all 25 decryptions of the
intercepted message. Read down the list and find the one that is English.

**Question 1:** which shift was used? Write out the hidden message.

*Your answer:*

In [ ]:
# TODO: print caesar(message, -shift) for every shift from 1 to 25

## Part 3: Break it automatically with frequency analysis (25 min)
Reading 25 lines is fine for one message. To break thousands you automate it. In English, `e` is the most common
letter, then `t`, `a`, `o`. The most common letter in the ciphertext is probably an encrypted `e`.

1. Count how often each letter appears in `message` (ignore case and non-letters).
2. Find the most common ciphertext letter.
3. Assume it maps to `e` and compute the shift. Decrypt with it.

Does your guessed shift match Part 2?

In [ ]:
counts = {}
# TODO: count each letter in message.lower() (letters only)

# TODO: find the most common letter, assume it is 'e', compute the shift
most_common = None
guess_shift = 0
print("Most common letter:", most_common, "-> guessed shift:", guess_shift)
print("Decrypted:", caesar(message, -guess_shift))

**Question 2:** frequency analysis guessed the key with no brute force. Why does this attack get *more*
reliable on longer messages, and when might it fail on a very short one?

*Your answer:*

## Part 4: Why modern encryption resists this (20 min)
The Caesar cipher fails because each letter always maps to the same letter, so the message keeps English's
letter-frequency fingerprint. Modern encryption like **AES** scrambles data so the output looks random: no
letter frequencies survive.

Run this. It uses the `cryptography` library's Fernet (AES under the hood). Look at how the ciphertext changes
completely when the message changes by one character, and that it is not readable text at all.

In [ ]:
from cryptography.fernet import Fernet

key = Fernet.generate_key()
f = Fernet(key)

c1 = f.encrypt(b"The meeting is at noon")
c2 = f.encrypt(b"The meeting is at noom")   # one letter different
print("Ciphertext 1:", c1[:50], "...")
print("Ciphertext 2:", c2[:50], "...")
print("Same length, totally different output.")

Now encrypt and decrypt a message of your own with Fernet. Fill in the two calls.

In [ ]:
secret_msg = b"Reserve the main hall for friday"
token = None      # TODO: encrypt secret_msg with f
print("Encrypted token:", token)

recovered = None  # TODO: decrypt token with f
print("Decrypted:", recovered.decode())
assert recovered == secret_msg

**Question 3:** try to decrypt `token` with a *different* key (make a new `Fernet(Fernet.generate_key())`
and call its `.decrypt`). What happens? Why is this the whole point of a secret key?

*Your answer:*

## Part 5: AI twist, spotting bad crypto code (25 min)
Developers often get encryption subtly wrong, and AI assistants will happily produce insecure code.
Here is a "password encryptor" an AI assistant wrote. Run it, then find the problems.

In [ ]:
# Written by an AI assistant: "encrypt a password so we can store it"
def ai_encrypt(password):
    key = "mysecretkey123"
    result = ""
    for i, ch in enumerate(password):
        result += chr(ord(ch) ^ ord(key[i % len(key)]))
    return result

print(repr(ai_encrypt("hunter2")))

Answer these, using what you learned this week. If your school allows it, also ask an AI assistant
"is there anything insecure about this code?" and compare its answer to yours.

1. This is XOR with a fixed, hardcoded key. Name two problems with hardcoding the key in the source.
2. Passwords should not be **encrypted** for storage at all. What should you do instead, and why?
   (Think back to Week 3.)
3. XOR with a repeating key keeps patterns, like Caesar did. If two users both have the password `aaaaaaa`,
   what do their "encrypted" values look like? Test it in the cell below.

*Your answers:*

In [ ]:
# TODO: call ai_encrypt("aaaaaaa") twice and compare the output

## Hands-on challenge: a secure note tool (15 min)
Build a tiny two-function tool a club could actually use to pass notes:

- `lock(message, key)`: returns an encrypted token (bytes). Use Fernet.
- `unlock(token, key)`: returns the original message (string), or the text `"WRONG KEY"` if the key is wrong.

*Hint:* a wrong key raises `cryptography.fernet.InvalidToken`; catch it with `try`/`except`.

In [ ]:
from cryptography.fernet import Fernet, InvalidToken

def lock(message, key):
    # TODO: encrypt message with Fernet(key)
    return b""

def unlock(token, key):
    # TODO: decrypt, but return "WRONG KEY" if the key is wrong
    return ""

good = Fernet.generate_key()
bad = Fernet.generate_key()
t = lock("meet at the library at 4", good)
print("With the right key:", unlock(t, good))
print("With the wrong key:", unlock(t, bad))

## Part 7: Reflect (10 min)
**Exit ticket.** Answer in 2 to 3 sentences each:

1. In your own words, why did the Caesar cipher fall to a computer in milliseconds, but AES does not?
2. You now have a tool that encrypts messages. Name one good use and one way someone could misuse encryption, and
   whether the math itself is "good" or "bad".

*Your answers:*

**Stretch (optional):** break a cipher where each letter maps to a *different* letter (a substitution cipher).
Why is brute force hopeless there (hint: 26! possibilities), and why does frequency analysis still work?